# Generation G - S1E? - Chain

This notebook is the companion of posts about Generative AI.

This episode shows how to use chains and memory

# Material

## Initializations

In [ ]:
### Update environment

In [ ]:
!apt-get update && apt-get install -y build-essential 1>/dev/null

In [ ]:
!apt-get update && apt-get install -y jq 1>/dev/null

In [ ]:
!pip install --upgrade pip  1>/dev/null

## Requirements

In [ ]:
#!pip install langchain==0.0.230 1>/dev/null
!pip install langchain==0.0.266 1>/dev/null

In [ ]:
!pip install openai==0.27.8 1>/dev/null

## Secrets and credentials

In [ ]:
%%bash --out secrets 
# using AWS's Secret Manager to store keys
# garb the keys and store it into a Pytthon variable
export RESPONSE=$(aws secretsmanager get-secret-value --secret-id 'salvia/labbench/tests' )
export SECRETS=$( echo $RESPONSE | jq '.SecretString | fromjson')

echo $SECRETS

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = eval(secrets)["OPENAI_API_KEY"]


# Code session

## custom criterion

In [ ]:
from langchain.evaluation.criteria import CriteriaEvalChain

llm = get_llm_model()

criteria = {"humor-criterion": "Is it funny?", "accuracy-criterion": "Is it accurate?"}  
evaluator = CriteriaEvalChain.from_llm(llm=llm, criteria=criteria)


1 request + 1 request per evaluator

## Labeled 

## Check against reference - labelled

In [ ]:
from langchain.llms import OpenAI
from langchain.evaluation.criteria import LabeledCriteriaEvalChain
from langchain.evaluation.criteria import CriteriaEvalChain

llm = get_llm_model()

criteria = "correctness"
eval_chain = LabeledCriteriaEvalChain.from_llm(
        llm=llm,
        criteria=criteria,
        requires_reference=True
    )

query = "What is the distance to the Moon?"
response = llm(query)
print(response)

evaluation = eval_chain.evaluate_strings(prediction=response, 
                                         input=query, 
                                         reference="384,000 km")
print(f"\n {pformat(evaluation)} \n")

## Check a bunch of answers - prepare datasets

## Check a bunch od answers - batch predict

In [ ]:
## Check a bunch od answers - batcheval

In [ ]:
%%time
from pprint import pformat
from langchain.llms import OpenAI
from langchain import LLMChain
from langchain.callbacks import get_openai_callback

from langchain.evaluation.criteria import LabeledCriteriaEvalChain
from langchain.evaluation.criteria import CriteriaEvalChain
from langchain.evaluation.qa import QAEvalChain

with get_openai_callback() as cb:

    qa_llm = get_llm_model()

    questions = [ qa['question'] for qa in questions_answers]
    print(f"\n questions \n {pformat(questions)} \n")

    results = llm.batch(questions)
    ## reshape the result so that it fits QAEval expectations
    predictions = [ {'question': qa['question'], 'answer': qa['answer'], 'result': r} 
                   for (qa, r) in zip(questions_answers, results)]         
    print(f"\n predictions \n {pformat(predictions)} \n")

    # Start your eval chain
    eval_llm = OpenAI(temperature=0.7)

    eval_chain = QAEvalChain.from_llm(llm)

    # Have it grade itself. The code below helps the eval_chain know where the different parts are
    graded_outputs = eval_chain.evaluate(questions_answers,
                                         predictions,
                                         question_key="question",
                                         prediction_key="result",
                                         answer_key='answer')

    print(f"\n graded output \n {pformat(graded_outputs)} \n")

    print(cb)
    print(f"token used={cb.total_tokens} total cost (USD)={cb.total_cost} \n")


# LLMs setup

In [ ]:
# cache reset
import langchain
from langchain.cache import InMemoryCache
langchain.llm_cache = InMemoryCache()

In [ ]:
from langchain.llms import OpenAI

# To make the caching really obvious, lets use a slower model.
llm = OpenAI(model_name="text-davinci-003", n=2, best_of=2)

In [ ]:
## chain

https://api.python.langchain.com/en/latest/chains/langchain.chains.llm.LLMChain.html#langchain.chains.llm.LLMChain

In [ ]:
from langchain.chat_models import ChatOpenAI

chatllm = ChatOpenAI(model_name="gpt-3.5-turbo")

In [ ]:
%%time
from langchain.callbacks import get_openai_callback

with get_openai_callback() as cb:
    query = "What is the distance to the Moon?"
    response = chatllm.predict(query)
    print(f"{response=}")
    print(f"\nUsage monitoring: token used={cb.total_tokens}, requests={cb.successful_requests}, total cost (USD)={cb.total_cost} \n")


In [ ]:
from langchain.prompts import PromptTemplate

# LLM chain consisting of the LLM and a prompt
prompt_template = """You are a teaching assistant. Your job is to write questions for an exam.
Given the following topic or topics, ask a question for each topic.
Topic: {topic}
Question: 
Relevant text, if any:"""
prompt = PromptTemplate(
    template=prompt_template, 
    input_variables=["topic"],    
)

In [ ]:
%%time
from langchain.callbacks import get_openai_callback
from langchain.chains import LLMChain

llm_chain = LLMChain(llm=llm, prompt=prompt)
                     #, output_key="question")

with get_openai_callback() as cb:
    topic = "The Moon"
    question = llm_chain.run(topic)
    print(f"{question=}")
    print(f"\nUsage monitoring: token used={cb.total_tokens}, requests={cb.successful_requests}, total cost (USD)={cb.total_cost} \n")


In [ ]:
%%time
from langchain.callbacks import get_openai_callback
from langchain.chains import LLMChain

chatllm_chain = LLMChain(llm=chatllm, prompt=prompt)

with get_openai_callback() as cb:
    topic = "The Moon"
    question = chatllm_chain(topic)
    print(f"{question=}")
    print(f"\nUsage monitoring: token used={cb.total_tokens}, requests={cb.successful_requests}, total cost (USD)={cb.total_cost} \n")


In [ ]:
%%time
from langchain.callbacks import get_openai_callback
from langchain.chains import LLMChain

chatllm_chain = LLMChain(llm=chatllm, prompt=prompt)

with get_openai_callback() as cb:
    topic = "The Moon"
    question = chatllm_chain.run(topic)   # similar but return the output string
    print(f"{question=}")
    print(f"\nUsage monitoring: token used={cb.total_tokens}, requests={cb.successful_requests}, total cost (USD)={cb.total_cost} \n")


In [ ]:
%%time
from langchain.callbacks import get_openai_callback
from langchain.chains import LLMChain

chatllm_chain = LLMChain(llm=chatllm, prompt=prompt)

with get_openai_callback() as cb:
    topic = "The Moon"
    question = chatllm_chain.predict(topic=topic)  # named parameters
    print(f"{question=}")
    print(f"\nUsage monitoring: token used={cb.total_tokens}, requests={cb.successful_requests}, total cost (USD)={cb.total_cost} \n")


In [ ]:
%%time
from pprint import pprint
from langchain.callbacks import get_openai_callback
from langchain.chains import LLMChain

chatllm_chain = LLMChain(llm=chatllm, prompt=prompt)

with get_openai_callback() as cb:
    topics = [
        {'topic': "The Moon"}, 
        {'topic': "Mars"}, 
        {'topic': "Jupiter"}, 
        {'topic': "Neptune"}, 
        {'topic': "Alpha centauri"}]
    questions = chatllm_chain.apply(topics)   # multiple inputs
    print("questions")
    pprint(questions)
    
    print(f"\nUsage monitoring: token used={cb.total_tokens}, requests={cb.successful_requests}, total cost (USD)={cb.total_cost} \n")


In [ ]:
%%time
from pprint import pprint
from langchain.callbacks import get_openai_callback
from langchain.chains import LLMChain

chatllm_chain = LLMChain(llm=chatllm, prompt=prompt)

with get_openai_callback() as cb:
    topics = [
        {'topic': "The Moon"}, 
        {'topic': "Mars"}, 
    ]
    results = chatllm_chain.generate(topics)   # multiple inputs + output as a result object
    print("== results")
    pprint(questions)
    print("== results generations")   
    pprint(results.generations)    
    print("== results llm_output")   
    pprint(results.llm_output)    
    print("== results run")   
    pprint(results.run)
    print(f"\nUsage monitoring: token used={cb.total_tokens}, requests={cb.successful_requests}, total cost (USD)={cb.total_cost} \n")


## options

usually documented in \_\_call\_\_

In [ ]:
%%time
from langchain.callbacks import get_openai_callback
from langchain.chains import LLMChain

chatllm_chain = LLMChain(llm=chatllm, prompt=prompt)

with get_openai_callback() as cb:
    topic = "The Moon"
    question = chatllm_chain(topic, return_only_outputs=False)  # True by default, add topic to the response
    print(f"{question=}")
    print(f"\nUsage monitoring: token used={cb.total_tokens}, requests={cb.successful_requests}, total cost (USD)={cb.total_cost} \n")


In [ ]:
%%time
from langchain.callbacks import get_openai_callback
from langchain.chains import LLMChain

chatllm_chain = LLMChain(llm=chatllm, prompt=prompt, verbose=True)

with get_openai_callback() as cb:
    topic = "The Moon"
    question = chatllm_chain(topic)  
    print(f"{question=}")
    print(f"\nUsage monitoring: token used={cb.total_tokens}, requests={cb.successful_requests}, total cost (USD)={cb.total_cost} \n")


In [ ]:
## memory

https://api.python.langchain.com/en/latest/memory/langchain.memory.simple.SimpleMemory.html

In [ ]:
%%time
from langchain.callbacks import get_openai_callback
from langchain.chains import LLMChain
from langchain.memory.simple import SimpleMemory

simple_memory = SimpleMemory()

chatllm_chain = LLMChain(llm=chatllm, prompt=prompt, memory=simple_memory)

with get_openai_callback() as cb:
    topic = "The Moon"
    question = chatllm_chain(topic)  
    print(f"{question=}")
    print(f"{simple_memory=}")
    print(f"\nUsage monitoring: token used={cb.total_tokens}, requests={cb.successful_requests}, total cost (USD)={cb.total_cost} \n")


In [ ]:
%%time
from langchain.callbacks import get_openai_callback
from langchain.chains import LLMChain
from langchain.memory.simple import SimpleMemory

simple_memory = SimpleMemory()

llm_chain = LLMChain(llm=llm, prompt=prompt, memory=simple_memory)

with get_openai_callback() as cb:
    topic = "The Moon"
    question = llm_chain(topic)  
    print(f"{question=}")
    print(f"{simple_memory=}")
    print(f"\nUsage monitoring: token used={cb.total_tokens}, requests={cb.successful_requests}, total cost (USD)={cb.total_cost} \n")


In [ ]:
%%time
from langchain.callbacks import get_openai_callback
from langchain.chains import LLMChain
from langchain.memory.simple import SimpleMemory

# load_memory_variables read r load ?

memories={"init": "test-123"}
simple_memory = SimpleMemory(memories=memories)

chatllm_chain = LLMChain(llm=chatllm, prompt=prompt, memory=simple_memory)

with get_openai_callback() as cb:
    topic = "The Moon"
    question = chatllm_chain(topic)  
    print(f"{question=}")
    print(f"{simple_memory=}")
    print(f"\nUsage monitoring: token used={cb.total_tokens}, requests={cb.successful_requests}, total cost (USD)={cb.total_cost} \n")


In [ ]:
## sequence

In [ ]:
%%time
from langchain.callbacks import get_openai_callback
from langchain.chains import LLMChain
from langchain.memory.simple import SimpleMemory
from langchain.chains.sequential import SimpleSequentialChain

# load_memory_variables read r load ?
simple_memory = SimpleMemory()

teacher_prompt_template = """You are a teaching assistant. Your job is to write questions for an exam.
Given the following topic or topics, ask a question for each topic.
Topic: {topic}
Question: 
Relevant text, if any:"""
teacher_prompt = PromptTemplate(
    template=teacher_prompt_template, 
    input_variables=["topic"],    
)
teacher_chain = LLMChain(llm=chatllm, prompt=teacher_prompt, memory=simple_memory)


student_prompt_template = """You are a student. Your job is to answer questions for an exam.
Given the following question or questions, give a reqponse for each question.
Question: {question}
Answer: 
Relevant text, if any:"""
student_prompt = PromptTemplate(
    template=student_prompt_template, 
    input_variables=["question"],    
)
student_chain = LLMChain(llm=chatllm, prompt=student_prompt, memory=simple_memory)

seq_chain = SimpleSequentialChain(chains=[teacher_chain, student_chain], memory=simple_memory)

with get_openai_callback() as cb:
    topic = "The Moon"
    answers = seq_chain(topic, return_only_outputs=False)  
    print(f"{answers=}")
    print(f"{simple_memory=}")
    print(f"\nUsage monitoring: token used={cb.total_tokens}, requests={cb.successful_requests}, total cost (USD)={cb.total_cost} \n")


https://api.python.langchain.com/en/latest/chains/langchain.chains.sequential.SequentialChain.html#langchain.chains.sequential.SequentialChain

In [ ]:
%%time
from langchain.callbacks import get_openai_callback
from langchain.chains import LLMChain
from langchain.memory.simple import SimpleMemory
from langchain.chains.sequential import SequentialChain

# load_memory_variables read r load ?
simple_memory = SimpleMemory()

teacher_prompt_template = """You are a teaching assistant. Your job is to write questions for an exam.
Given the following topic or topics, ask a question for each topic.
Topic: {topic}
Question: 
Relevant text, if any:"""
teacher_prompt = PromptTemplate(
    template=teacher_prompt_template, 
    input_variables=["topic"],    
)
teacher_chain = LLMChain(llm=chatllm, 
    prompt=teacher_prompt, 
    output_key="question", 
    memory=simple_memory,
    verbose=True)


student_prompt_template = """You are a student. Your job is to answer questions for an exam.
Given the following question or questions, give a reqponse for each question.
Topic: {topic}
Question: {question}
Answer: 
Relevant text, if any:"""
student_prompt = PromptTemplate(
    template=student_prompt_template, 
    input_variables=["topic", "question"],    
)
student_chain = LLMChain(llm=chatllm, 
    prompt=student_prompt, 
    output_key="answer", 
    memory=simple_memory, 
    verbose=True)
# must define output_key to avoid Chain returned keys that already exist: {'text'} (type=value_error)

seq_chain = SequentialChain(chains=[
    teacher_chain, student_chain], 
    input_variables=["topic"], 
    memory=simple_memory,
    verbose=True)

with get_openai_callback() as cb:
    topic = "The Moon"
    answers = seq_chain(topic)  
    print(f"{answers=}")
    print(f"{simple_memory=}")
    print(f"\nUsage monitoring: token used={cb.total_tokens}, requests={cb.successful_requests}, total cost (USD)={cb.total_cost} \n")


In [ ]:
%%time
from langchain.callbacks import get_openai_callback
from langchain.chains import LLMChain
from langchain.memory.simple import SimpleMemory
from langchain.chains.sequential import SequentialChain

# load_memory_variables read r load ?
memories={"context": "exam 123"}
simple_memory = SimpleMemory(memories=memories)

teacher_prompt_template = """You are a teaching assistant. Your job is to write questions for an exam.
Given the following topic or topics, ask a question for each topic.
Topic: {topic}
Question: 
Relevant text, if any:"""
teacher_prompt = PromptTemplate(
    template=teacher_prompt_template, 
    input_variables=["topic"],    
)
teacher_chain = LLMChain(llm=chatllm, 
    prompt=teacher_prompt, 
    output_key="question", 
    memory=simple_memory,
    verbose=True)


student_prompt_template = """You are a student. Your job is to answer questions for an exam.
Given the following question or questions, give a reqponse for each question.
Include the question and the axam name in the response. 
Exam name: {context}
Topic: {topic}
Question: {question}
Answer: 
Relevant text, if any:"""
student_prompt = PromptTemplate(
    template=student_prompt_template, 
    input_variables=["topic", "question", "context"],    
)
student_chain = LLMChain(llm=chatllm, 
    prompt=student_prompt, 
    output_key="answer", 
    memory=simple_memory, 
    verbose=True)
# must define output_key to avoid Chain returned keys that already exist: {'text'} (type=value_error)

seq_chain = SequentialChain(chains=[
    teacher_chain, student_chain], 
    input_variables=["topic"], 
    memory=simple_memory,
    verbose=True)

with get_openai_callback() as cb:
    topic = "The Moon"
    answers = seq_chain(topic)  
    print(f"{answers=}")
    print(f"{simple_memory=}")
    print(f"\nUsage monitoring: token used={cb.total_tokens}, requests={cb.successful_requests}, total cost (USD)={cb.total_cost} \n")


In [ ]:
## 